# LeetCode 3105 — Longest Strictly Increasing or Strictly Decreasing Subarray bằng Python

Link đề: https://leetcode.com/problems/longest-strictly-increasing-or-strictly-decreasing-subarray/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def longestMonotonicSubarray(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **độ dài lớn nhất** của một **mảng con liên tiếp** mà nó **tăng ngặt** (`a < b < c ...`) **hoặc giảm ngặt** (`a > b > c ...`).
2. Phân biệt **mảng con (subarray, liên tiếp)** với **dãy con (subsequence, không cần liên tiếp)** và hiểu "**ngặt**" nghĩa là **bằng nhau không được chấp nhận**.
3. Đi từ cách **kiểm tra mọi mảng con** → **hai lượt: một lượt tìm chuỗi tăng dài nhất, một lượt tìm chuỗi giảm dài nhất** → **một lượt với hai bộ đếm `inc`, `dec`** → **dùng "hướng" của bước nhảy** → **`itertools.groupby` trên dấu của hiệu hai phần tử liền kề**.
4. Hiểu mấu chốt: chỉ cần so **hai phần tử liền kề**; một **đoạn tăng ngặt** kéo dài khi `nums[i] > nums[i-1]` và **bị cắt** khi không; tương tự cho đoạn giảm; hai đoạn **không** thể "nối" với nhau ở điểm đổi hướng.
5. Tránh các lỗi kinh điển: dùng `>=` / `<=` (không ngặt), **không reset về `1`** khi gặp phần tử bằng nhau, khởi tạo bộ đếm bằng `0` thay vì `1`, quên cập nhật đáp án ở đoạn cuối, nhầm mảng con với dãy con.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums`. Hãy trả về **độ dài của mảng con dài nhất** thỏa **một trong hai điều kiện**:

- **tăng ngặt** (strictly increasing): mỗi phần tử **lớn hơn hẳn** phần tử đứng trước;
- hoặc **giảm ngặt** (strictly decreasing): mỗi phần tử **nhỏ hơn hẳn** phần tử đứng trước.

(Mảng con là dãy các phần tử **liên tiếp**. Một phần tử đơn lẻ vừa là tăng ngặt vừa là giảm ngặt.)

**Ví dụ 1**

- Input: `nums = [1, 4, 3, 3, 2]`
- Output: `2`
- Giải thích: các mảng con tăng ngặt dài nhất là `[1]`, `[2]`, `[3]`, `[4]`, `[1, 4]` (dài `2`); các mảng con giảm ngặt dài nhất là `[1]`, `[2]`, `[3]`, `[4]`, `[3, 2]`, `[4, 3]` (dài `2`). Mảng con `[4, 3, 3]` **không** giảm ngặt vì `3` không nhỏ hơn hẳn `3`. Đáp án `2`.

**Ví dụ 2**

- Input: `nums = [3, 3, 3, 3]`
- Output: `1`
- Giải thích: mọi phần tử bằng nhau nên không có mảng con nào dài hơn `1` thỏa tăng ngặt hay giảm ngặt.

**Ví dụ 3**

- Input: `nums = [3, 2, 1]`
- Output: `3`
- Giải thích: cả mảng `[3, 2, 1]` giảm ngặt, độ dài `3`.

**Ràng buộc:** `1 <= nums.length <= 50`; `1 <= nums[i] <= 50`.

In [1]:
from typing import List
from itertools import groupby
import random

examples = [
    ([1, 4, 3, 3, 2], 2),
    ([3, 3, 3, 3], 1),
    ([3, 2, 1], 3),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 4, 3, 3, 2] -> kỳ vọng 2
nums = [3, 3, 3, 3] -> kỳ vọng 1
nums = [3, 2, 1] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Mảng con và dãy con khác nhau

**Mảng con (subarray)** là các phần tử **liền nhau** trong mảng gốc; **dãy con (subsequence)** có thể bỏ qua phần tử. Với `[1, 4, 3, 3, 2]`: `[4, 3]` là mảng con (chỉ số `1..2`); `[1, 3, 2]` là dãy con nhưng **không** là mảng con. Bài này hỏi **mảng con**, nên chỉ so sánh các phần tử **liền kề**.

In [2]:
nums = [1, 4, 3, 3, 2]
print(nums[1:3])                       # [4, 3]  mảng con (liền nhau)
print([nums[0], nums[2], nums[4]])     # [1, 3, 2] dãy con (bỏ qua phần tử): KHÔNG phải mảng con

[4, 3]
[1, 3, 2]


### 3.2. "Ngặt" và "không ngặt"

- **Tăng ngặt:** `a < b` (bằng nhau thì **không** tính).
- **Tăng không ngặt:** `a <= b`.

Chỉ một cặp bằng nhau liền kề (như `3, 3`) đã **cắt đứt** cả đoạn tăng ngặt lẫn đoạn giảm ngặt.

In [3]:
print(3 < 3, 3 <= 3)       # False True
print(all(a < b for a, b in zip([1, 2, 3], [2, 3, 4])))     # True  [1,2,3,4] tăng ngặt
print(all(a < b for a, b in zip([1, 2, 2], [2, 2, 3])))     # False [1,2,2,3] có cặp bằng nhau

False True
True
False


### 3.3. Duyệt hai phần tử liền kề: `zip(nums, nums[1:])` và chỉ số

Để so `nums[i - 1]` với `nums[i]` ta có thể:

- dùng chỉ số: `for i in range(1, len(nums))` rồi so `nums[i - 1]`, `nums[i]`;
- dùng `zip(nums, nums[1:])`: ghép từng phần tử với phần tử kế tiếp.

In [4]:
nums = [1, 4, 3, 3, 2]
for i in range(1, len(nums)):
    print(i, nums[i - 1], "->", nums[i])
print(list(zip(nums, nums[1:])))        # [(1, 4), (4, 3), (3, 3), (3, 2)]

1 1 -> 4
2 4 -> 3
3 3 -> 3
4 3 -> 2
[(1, 4), (4, 3), (3, 3), (3, 2)]


### 3.4. Đoạn (run) và bộ đếm có reset

Một **đoạn tăng ngặt** là dãy `nums[l..r]` mà mỗi cặp liền kề đều tăng. Ta có thể đếm chiều dài đoạn **đang chạy** bằng biến `inc`:

- nếu `nums[i] > nums[i-1]`: `inc += 1` (đoạn kéo dài);
- ngược lại: `inc = 1` (đoạn mới bắt đầu tại `nums[i]`, độ dài `1`).

Bắt đầu `inc = 1` vì phần tử đầu tiên tự nó là một đoạn dài `1`.

In [5]:
nums = [1, 2, 3, 1, 2]
inc = 1
for i in range(1, len(nums)):
    inc = inc + 1 if nums[i] > nums[i - 1] else 1
    print(f"i={i}: {nums[i - 1]} -> {nums[i]}, inc={inc}")

i=1: 1 -> 2, inc=2
i=2: 2 -> 3, inc=3
i=3: 3 -> 1, inc=1
i=4: 1 -> 2, inc=2


### 3.5. `itertools.groupby`: gom các phần tử liền kề giống nhau

`groupby(iterable)` gom **các phần tử liên tiếp có cùng giá trị** thành từng nhóm `(giá_trị, nhóm)`. Với danh sách "dấu của hiệu hai phần tử liền kề" (`+1`, `-1`, `0`) ta đếm được độ dài mỗi đoạn có cùng hướng. Cần chuyển `nhóm` thành list để lấy độ dài (`len(list(nhóm))`).

In [6]:
signs = [1, 1, -1, 0, 0, -1]
for key, grp in groupby(signs):
    print(key, len(list(grp)))        # 1 2 / -1 1 / 0 2 / -1 1

1 2
-1 1
0 2
-1 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [1, 4, 3, 3, 2]`. Ta nhìn **hướng của từng bước** (so với phần tử đứng trước):

| bước | cặp | hướng |
|---|---|---|
| 1 | `1 → 4` | tăng |
| 2 | `4 → 3` | giảm |
| 3 | `3 → 3` | bằng |
| 4 | `3 → 2` | giảm |

Một đoạn tăng/giảm ngặt dài nhất là một **đoạn liên tiếp các bước cùng hướng (tăng hoặc giảm), không có bước "bằng"**. Ở đây: bước `1` tăng (một bước ⇒ đoạn dài `2` phần tử: `[1, 4]`); bước `2` giảm một mình (đoạn `[4, 3]`); bước `3` bằng (cắt); bước `4` giảm một mình (đoạn `[3, 2]`). Mỗi đoạn có `1` bước ⇒ `2` phần tử. Đáp án `2`.

Quy tắc: **đoạn gồm `s` bước cùng hướng ngặt ⇒ độ dài `s + 1` phần tử**; nếu mọi bước đều "bằng" thì đáp án `1`.

In [7]:
nums = [1, 4, 3, 3, 2]
dirs = []
for i in range(1, len(nums)):
    if nums[i] > nums[i - 1]:
        dirs.append("tăng")
    elif nums[i] < nums[i - 1]:
        dirs.append("giảm")
    else:
        dirs.append("bằng")
print(dirs)               # ['tăng', 'giảm', 'bằng', 'giảm']

['tăng', 'giảm', 'bằng', 'giảm']


## 5. Cách 1 — Brute Force / baseline: kiểm tra mọi mảng con

**Ý tưởng.** Với mọi cặp `(i, j)`, `i <= j`: lấy `sub = nums[i..j]`, kiểm tra `sub` có tăng ngặt hay giảm ngặt không (so mọi cặp liền kề), nếu có thì cập nhật đáp án bằng `j - i + 1`.

In [8]:
def is_strictly_increasing(sub: List[int]) -> bool:
    return all(sub[t] < sub[t + 1] for t in range(len(sub) - 1))


def is_strictly_decreasing(sub: List[int]) -> bool:
    return all(sub[t] > sub[t + 1] for t in range(len(sub) - 1))


def longest_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    best = 1                                           # mảng không rỗng: ít nhất 1 phần tử
    for i in range(n):
        for j in range(i, n):
            sub = nums[i:j + 1]
            if is_strictly_increasing(sub) or is_strictly_decreasing(sub):
                best = max(best, j - i + 1)
    return best


for nums, expected in examples:
    print(nums, longest_bruteforce(nums), "(kỳ vọng", expected, ")")

[1, 4, 3, 3, 2] 2 (kỳ vọng 2 )
[3, 3, 3, 3] 1 (kỳ vọng 1 )
[3, 2, 1] 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Có `O(n²)` mảng con, mỗi lần kiểm tra mất `O(n)` ⇒ **Time `O(n³)`**, **Space `O(n)`** (lát cắt). Với `n <= 50` chạy được (khoảng `10^5` bước), nhưng lãng phí.

**Nút thắt:** việc kiểm tra `nums[i..j]` lặp lại phần kiểm tra của `nums[i..j-1]`; mỗi lần chỉ thêm **một cặp liền kề mới**.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát.** Mảng con `nums[l..r]` tăng ngặt ⇔ **mọi cặp liền kề** `(nums[t-1], nums[t])` với `t` từ `l + 1` đến `r` đều tăng. Nếu ta biết độ dài đoạn tăng ngặt **kết thúc tại `t - 1`** là `inc`, thì:

- nếu `nums[t] > nums[t - 1]`: đoạn kết thúc tại `t` dài `inc + 1`;
- nếu không: đoạn kết thúc tại `t` chỉ gồm `nums[t]`, dài `1`.

Tương tự với đoạn giảm ngặt (`dec`). Ta chỉ cần **hai biến** chạy dọc mảng, và đáp án là `max` mọi giá trị `inc`, `dec` từng gặp. Một lượt duyệt: **`O(n)`**.

In [9]:
# Quy tắc cập nhật trên [1, 4, 3, 3, 2]
nums = [1, 4, 3, 3, 2]
inc = dec = 1
best = 1
for i in range(1, len(nums)):
    inc = inc + 1 if nums[i] > nums[i - 1] else 1
    dec = dec + 1 if nums[i] < nums[i - 1] else 1
    best = max(best, inc, dec)
    print(f"i={i}: {nums[i - 1]} -> {nums[i]}, inc={inc}, dec={dec}, best={best}")

i=1: 1 -> 4, inc=2, dec=1, best=2
i=2: 4 -> 3, inc=1, dec=2, best=2
i=3: 3 -> 3, inc=1, dec=1, best=2
i=4: 3 -> 2, inc=1, dec=2, best=2


## 7. Các cách tiếp theo

### Cách 2 — Hai lượt riêng: chuỗi tăng dài nhất rồi chuỗi giảm dài nhất

**Giải thích.** Viết một hàm tìm **đoạn tăng ngặt dài nhất**; để tìm **đoạn giảm ngặt dài nhất**, ta tận dụng ý tưởng "đảo dấu": mảng `nums` giảm ngặt ⇔ mảng `[-x for x in nums]` tăng ngặt. Đáp án là `max` của hai kết quả. Cách này tái sử dụng một hàm duy nhất.

In [10]:
def longest_increasing_run(nums: List[int]) -> int:
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i] > nums[i - 1] else 1     # kéo dài hoặc bắt đầu lại
        best = max(best, cur)
    return best


def longest_two_passes(nums: List[int]) -> int:
    inc = longest_increasing_run(nums)
    dec = longest_increasing_run([-x for x in nums])      # giảm ngặt = tăng ngặt sau khi đảo dấu
    return max(inc, dec)


for nums, expected in examples:
    print(nums, longest_two_passes(nums), "(kỳ vọng", expected, ")")

[1, 4, 3, 3, 2] 2 (kỳ vọng 2 )
[3, 3, 3, 3] 1 (kỳ vọng 1 )
[3, 2, 1] 3 (kỳ vọng 3 )


**Độ phức tạp Cách 2:** hai lượt duyệt `O(n)` + tạo mảng đảo dấu `O(n)` ⇒ **Time `O(n)`**, **Space `O(n)`**. (Có thể tránh mảng phụ bằng cách truyền hàm so sánh.)

### Cách 3 — Một lượt, hai bộ đếm `inc` và `dec`

**Giải thích.** Gộp hai lượt thành một: mỗi bước cập nhật `inc` (đoạn tăng đang chạy) và `dec` (đoạn giảm đang chạy), rồi `best = max(best, inc, dec)`. Nhớ: khi `nums[i] == nums[i-1]` **cả hai** đều reset về `1`.

In [11]:
def longest_one_pass(nums: List[int]) -> int:
    inc = dec = 1                                    # đoạn đang chạy kết thúc tại phần tử hiện tại
    best = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            inc += 1                                 # đoạn tăng kéo dài
            dec = 1                                  # đoạn giảm bị cắt
        elif nums[i] < nums[i - 1]:
            dec += 1                                 # đoạn giảm kéo dài
            inc = 1                                  # đoạn tăng bị cắt
        else:
            inc = dec = 1                            # bằng nhau: cắt cả hai
        best = max(best, inc, dec)
    return best


for nums, expected in examples:
    print(nums, longest_one_pass(nums), "(kỳ vọng", expected, ")")

[1, 4, 3, 3, 2] 2 (kỳ vọng 2 )
[3, 3, 3, 3] 1 (kỳ vọng 1 )
[3, 2, 1] 3 (kỳ vọng 3 )


In [12]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionOnePass:
    def longestMonotonicSubarray(self, nums: List[int]) -> int:
        inc = dec = best = 1
        for i in range(1, len(nums)):
            if nums[i] > nums[i - 1]:
                inc, dec = inc + 1, 1
            elif nums[i] < nums[i - 1]:
                inc, dec = 1, dec + 1
            else:
                inc = dec = 1
            best = max(best, inc, dec)
        return best


print(SolutionOnePass().longestMonotonicSubarray([1, 4, 3, 3, 2]))     # 2

2


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Một biến độ dài và một biến "hướng" (state machine)

**Giải thích.** Thay vì hai bộ đếm, ta giữ **hướng hiện tại** `d` (`+1` tăng, `-1` giảm, `0` chưa có hướng) và độ dài `length` của đoạn hiện tại. Mỗi bước tính `s = dấu(nums[i] - nums[i-1])`:

- `s == 0`: reset `length = 1`, `d = 0`;
- `s == d`: cùng hướng ⇒ `length += 1`;
- `s != d` (kể cả khi `d == 0`): đổi hướng ⇒ đoạn mới gồm **hai** phần tử `nums[i-1], nums[i]` ⇒ `length = 2`, `d = s`.

Khi đổi hướng, đoạn mới bắt đầu từ **phần tử đứng trước** (điểm đổi hướng thuộc cả hai đoạn).

In [13]:
def sign(a: int) -> int:
    return (a > 0) - (a < 0)                          # +1, 0, hoặc -1


def longest_state(nums: List[int]) -> int:
    best = length = 1
    d = 0                                             # hướng hiện tại: +1 tăng, -1 giảm, 0 chưa có
    for i in range(1, len(nums)):
        s = sign(nums[i] - nums[i - 1])
        if s == 0:
            length, d = 1, 0                          # bằng nhau: cắt
        elif s == d:
            length += 1                               # cùng hướng
        else:
            length, d = 2, s                          # đổi hướng: đoạn mới gồm nums[i-1], nums[i]
        best = max(best, length)
    return best


for nums, expected in examples:
    print(nums, longest_state(nums), "(kỳ vọng", expected, ")")

[1, 4, 3, 3, 2] 2 (kỳ vọng 2 )
[3, 3, 3, 3] 1 (kỳ vọng 1 )
[3, 2, 1] 3 (kỳ vọng 3 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**. Ưu điểm: dễ mở rộng (đếm số lần đổi hướng, ...); nhược điểm: chỗ "đổi hướng ⇒ `length = 2`" dễ viết sai.

### Cách 5 — `groupby` trên dấu của các hiệu liền kề

**Giải thích.** Tính `signs[t] = sign(nums[t+1] - nums[t])` cho `t = 0..n-2`. Mỗi đoạn tối đa các dấu **liên tiếp bằng nhau và khác `0`** có độ dài `m` tương ứng một đoạn đơn điệu ngặt có `m + 1` phần tử. `groupby(signs)` cho ta các đoạn đó trực tiếp; bỏ nhóm có dấu `0`. Nếu không có nhóm nào (mọi dấu là `0`, hoặc `n = 1`) thì đáp án `1`.

In [14]:
def longest_groupby(nums: List[int]) -> int:
    signs = [sign(b - a) for a, b in zip(nums, nums[1:])]     # dấu của hiệu hai phần tử liền kề
    best = 1
    for s, grp in groupby(signs):
        if s != 0:
            best = max(best, len(list(grp)) + 1)              # m bước cùng hướng -> m + 1 phần tử
    return best


for nums, expected in examples:
    print(nums, longest_groupby(nums), "(kỳ vọng", expected, ")")

[1, 4, 3, 3, 2] 2 (kỳ vọng 2 )
[3, 3, 3, 3] 1 (kỳ vọng 1 )
[3, 2, 1] 3 (kỳ vọng 3 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (mảng dấu). Rất "Pythonic" nhưng tốn bộ nhớ phụ.

## 8. TRACE từng bước

Ta trace Cách 3 trên `nums = [1, 4, 3, 3, 2]` và Cách 4 trên `nums = [1, 2, 3, 2, 1, 0, 0, 5]` để thấy các lần đổi hướng và reset.

In [15]:
def trace_one_pass(nums: List[int]) -> int:
    inc = dec = best = 1
    print("nums =", nums)
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if b > a:
            inc, dec = inc + 1, 1
            why = "tăng: inc kéo dài, dec reset"
        elif b < a:
            inc, dec = 1, dec + 1
            why = "giảm: dec kéo dài, inc reset"
        else:
            inc = dec = 1
            why = "bằng: reset cả hai"
        best = max(best, inc, dec)
        print(f"i={i}: {a} -> {b} ({why}) | inc={inc}, dec={dec}, best={best}")
    return best


assert trace_one_pass([1, 4, 3, 3, 2]) == 2

nums = [1, 4, 3, 3, 2]
i=1: 1 -> 4 (tăng: inc kéo dài, dec reset) | inc=2, dec=1, best=2
i=2: 4 -> 3 (giảm: dec kéo dài, inc reset) | inc=1, dec=2, best=2
i=3: 3 -> 3 (bằng: reset cả hai) | inc=1, dec=1, best=2
i=4: 3 -> 2 (giảm: dec kéo dài, inc reset) | inc=1, dec=2, best=2


In [16]:
def trace_state(nums: List[int]) -> int:
    best = length = 1
    d = 0
    print("nums =", nums)
    for i in range(1, len(nums)):
        s = sign(nums[i] - nums[i - 1])
        if s == 0:
            length, d = 1, 0
            note = "bằng: reset"
        elif s == d:
            length += 1
            note = "cùng hướng: kéo dài"
        else:
            length, d = 2, s
            note = "đổi hướng: đoạn mới gồm 2 phần tử"
        best = max(best, length)
        print(f"i={i}: {nums[i - 1]} -> {nums[i]} s={s:+d} d={d:+d} ({note}) | length={length}, best={best}")
    return best


assert trace_state([1, 2, 3, 2, 1, 0, 0, 5]) == 4

nums = [1, 2, 3, 2, 1, 0, 0, 5]
i=1: 1 -> 2 s=+1 d=+1 (đổi hướng: đoạn mới gồm 2 phần tử) | length=2, best=2
i=2: 2 -> 3 s=+1 d=+1 (cùng hướng: kéo dài) | length=3, best=3
i=3: 3 -> 2 s=-1 d=-1 (đổi hướng: đoạn mới gồm 2 phần tử) | length=2, best=3
i=4: 2 -> 1 s=-1 d=-1 (cùng hướng: kéo dài) | length=3, best=3
i=5: 1 -> 0 s=-1 d=-1 (cùng hướng: kéo dài) | length=4, best=4
i=6: 0 -> 0 s=+0 d=+0 (bằng: reset) | length=1, best=4
i=7: 0 -> 5 s=+1 d=+1 (đổi hướng: đoạn mới gồm 2 phần tử) | length=2, best=4


Bảng trace Cách 3 với `[1, 4, 3, 3, 2]`:

| i | cặp | inc | dec | best |
|---|---|---|---|---|
| 1 | 1 → 4 (tăng) | 2 | 1 | 2 |
| 2 | 4 → 3 (giảm) | 1 | 2 | 2 |
| 3 | 3 → 3 (bằng) | 1 | 1 | 2 |
| 4 | 3 → 2 (giảm) | 1 | 2 | 2 |

Đáp án `2`. Với `[1, 2, 3, 2, 1, 0, 0, 5]` các đoạn dài nhất là `[1, 2, 3]` (tăng, dài 3) và `[3, 2, 1, 0]` (giảm, dài 4) ⇒ đáp án `4`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (một lượt, hai bộ đếm): ngắn, dễ nhớ, `O(n)` thời gian và `O(1)` bộ nhớ.

In [17]:
from typing import List

class Solution:
    def longestMonotonicSubarray(self, nums: List[int]) -> int:
        inc = dec = best = 1                       # độ dài đoạn tăng / giảm đang chạy, và đáp án
        for i in range(1, len(nums)):
            if nums[i] > nums[i - 1]:
                inc, dec = inc + 1, 1              # tăng: kéo dài đoạn tăng, cắt đoạn giảm
            elif nums[i] < nums[i - 1]:
                inc, dec = 1, dec + 1              # giảm: kéo dài đoạn giảm, cắt đoạn tăng
            else:
                inc = dec = 1                      # bằng nhau: cắt cả hai
            best = max(best, inc, dec)
        return best


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.longestMonotonicSubarray(nums))

[1, 4, 3, 3, 2] -> 2
[3, 3, 3, 3] -> 1
[3, 2, 1] -> 3


## 10. Vì sao thuật toán đúng

- **Bất biến:** sau khi xử lý phần tử `i`, `inc` bằng độ dài **đoạn tăng ngặt dài nhất kết thúc tại `i`**, và `dec` tương tự cho giảm. Chứng minh quy nạp: ở `i = 0` cả hai bằng `1`. Giả sử đúng tại `i - 1`; nếu `nums[i] > nums[i-1]` thì đoạn tăng kết thúc tại `i` là đoạn tăng kết thúc tại `i - 1` kéo thêm `nums[i]` (độ dài `inc + 1`), còn đoạn giảm kết thúc tại `i` chỉ có thể là `[nums[i]]` (vì `nums[i-1] < nums[i]` phá giảm) nên `dec = 1`. Các nhánh còn lại tương tự.
- **Không bỏ sót:** mọi mảng con đơn điệu ngặt kết thúc tại một chỉ số `i` nào đó; đoạn dài nhất kết thúc tại `i` được `inc` hoặc `dec` giữ, và `best` lấy `max` qua mọi `i`.
- **Phần tử bằng nhau cắt cả hai đoạn:** vì đề yêu cầu **ngặt**, hai phần tử bằng nhau liền kề không thể cùng nằm trong một đoạn tăng ngặt hay giảm ngặt.
- **`n = 1`:** vòng `for` không chạy, trả `1` (đúng, mảng một phần tử).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Kiểm tra mọi mảng con | `O(n³)` | `O(n)` |
| 2. Hai lượt (đảo dấu) | `O(n)` | `O(n)` |
| 3. Một lượt, `inc` và `dec` | `O(n)` | `O(1)` |
| 4. Trạng thái hướng | `O(n)` | `O(1)` |
| 5. `groupby` trên dấu | `O(n)` | `O(n)` |

Phải đọc mọi phần tử nên `O(n)` thời gian là tối ưu.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "brute": longest_bruteforce,
    "two_pass": longest_two_passes,
    "one_pass": longest_one_pass,
    "state": longest_state,
    "groupby": longest_groupby,
    "leetcode": sol.longestMonotonicSubarray,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        data = list(nums)
        actual = fn(data)
        assert actual == expected, (label, key, nums, actual, expected)
        assert data == nums, ("input bị sửa", key)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all([5], 1, "một phần tử")
check_all([1, 2], 2, "hai phần tử tăng")
check_all([2, 1], 2, "hai phần tử giảm")
check_all([2, 2], 1, "hai phần tử bằng nhau")
check_all([1, 2, 3, 4, 5], 5, "tăng ngặt toàn bộ")
check_all([5, 4, 3, 2, 1], 5, "giảm ngặt toàn bộ")
check_all([1, 2, 3, 2, 1], 3, "tăng rồi giảm: điểm đổi hướng thuộc cả hai đoạn, nhưng đoạn dài nhất là 3")
check_all([3, 2, 1, 2, 3], 3, "giảm rồi tăng")
check_all([1, 2, 2, 3], 2, "phần tử bằng cắt đoạn")
check_all([1, 3, 2, 4, 3, 5], 2, "zigzag: mọi đoạn dài 2")
check_all([1, 2, 3, 3, 4, 5, 6], 4, "hai đoạn tăng, đoạn sau dài hơn")
check_all(list(range(1, 51)), 50, "n = 50 tăng ngặt")
check_all(list(range(50, 0, -1)), 50, "n = 50 giảm ngặt")
check_all([7] * 50, 1, "n = 50 toàn bằng nhau")

PASS một phần tử
PASS hai phần tử tăng
PASS hai phần tử giảm
PASS hai phần tử bằng nhau
PASS tăng ngặt toàn bộ
PASS giảm ngặt toàn bộ
PASS tăng rồi giảm: điểm đổi hướng thuộc cả hai đoạn, nhưng đoạn dài nhất là 3
PASS giảm rồi tăng
PASS phần tử bằng cắt đoạn
PASS zigzag: mọi đoạn dài 2
PASS hai đoạn tăng, đoạn sau dài hơn
PASS n = 50 tăng ngặt
PASS n = 50 giảm ngặt
PASS n = 50 toàn bằng nhau


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **dùng `>=`** (không ngặt); (b) **chỉ xét tăng** hoặc **không reset khi bằng nhau**.

In [20]:
def wrong_non_strict(nums):
    inc = dec = best = 1
    for i in range(1, len(nums)):
        inc = inc + 1 if nums[i] >= nums[i - 1] else 1        # SAI: không ngặt
        dec = dec + 1 if nums[i] <= nums[i - 1] else 1
        best = max(best, inc, dec)
    return best


def wrong_only_increasing(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i] > nums[i - 1] else 1
        best = max(best, cur)
    return best                                                # SAI: bỏ qua giảm


def wrong_no_reset_on_equal(nums):
    inc = dec = best = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            inc, dec = inc + 1, 1
        elif nums[i] < nums[i - 1]:
            inc, dec = 1, dec + 1
        best = max(best, inc, dec)                             # SAI: bằng nhau không reset
    return best


print(wrong_non_strict([3, 3, 3, 3]), "!=", longest_one_pass([3, 3, 3, 3]))
print(wrong_only_increasing([3, 2, 1]), "!=", longest_one_pass([3, 2, 1]))
print(wrong_no_reset_on_equal([3, 2, 2, 1]), "!=", longest_one_pass([3, 2, 2, 1]))
assert wrong_non_strict([3, 3, 3, 3]) != longest_one_pass([3, 3, 3, 3])
assert wrong_only_increasing([3, 2, 1]) != longest_one_pass([3, 2, 1])
assert wrong_no_reset_on_equal([3, 2, 2, 1]) != longest_one_pass([3, 2, 2, 1])

4 != 1
1 != 3
3 != 2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh input đúng ràng buộc (`n <= 50`, giá trị `1..50`), gồm nhiều dạng: giá trị nhỏ (nhiều bằng nhau), tăng/giảm dần có nhiễu, ngẫu nhiên đều. Cách 1 là chuẩn.

In [21]:
def random_nums(rng):
    n = rng.choice([1, 2, 3, 5, 10, 25, 50])
    mode = rng.choice(["tiny", "random", "trend_up", "trend_down", "zigzag"])
    if mode == "tiny":
        return [rng.randint(1, 3) for _ in range(n)]
    if mode == "random":
        return [rng.randint(1, 50) for _ in range(n)]
    if mode == "trend_up":
        v, out = 1, []
        for _ in range(n):
            out.append(v)
            v = min(50, max(1, v + rng.choice([-1, 1, 1, 1, 0])))
        return out
    if mode == "trend_down":
        v, out = 50, []
        for _ in range(n):
            out.append(v)
            v = min(50, max(1, v + rng.choice([1, -1, -1, -1, 0])))
        return out
    return [rng.choice([10, 20]) + rng.choice([0, 1]) for _ in range(n)]


rng = random.Random(3105)
for trial in range(3000):
    nums = random_nums(rng)
    assert 1 <= len(nums) <= 50 and all(1 <= v <= 50 for v in nums)
    ref = longest_bruteforce(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 input ngẫu nhiên đúng ràng buộc: mọi cách khớp brute force")

PASS 3000 input ngẫu nhiên đúng ràng buộc: mọi cách khớp brute force


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` / `<=` (không ngặt)

In [22]:
def sai_khong_ngat(nums):
    inc = dec = best = 1
    for i in range(1, len(nums)):
        inc = inc + 1 if nums[i] >= nums[i - 1] else 1
        dec = dec + 1 if nums[i] <= nums[i - 1] else 1
        best = max(best, inc, dec)
    return best


print("sai:", sai_khong_ngat([3, 3, 3, 3]), "| đúng:", Solution().longestMonotonicSubarray([3, 3, 3, 3]))

sai: 4 | đúng: 1


**Vì sao sai:** đề yêu cầu **ngặt**: hai phần tử bằng nhau **không** thỏa. Với `[3, 3, 3, 3]` code sai cho `4` thay vì `1`. **Cách sửa:** dùng `>` và `<`.

### Lỗi 2: Không reset khi hai phần tử bằng nhau (hoặc chỉ reset một bộ đếm)

In [23]:
def sai_khong_reset(nums):
    inc = dec = best = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            inc, dec = inc + 1, 1
        elif nums[i] < nums[i - 1]:
            inc, dec = 1, dec + 1
        # thiếu nhánh "bằng nhau": inc, dec không bị reset
        best = max(best, inc, dec)
    return best


print("sai:", sai_khong_reset([3, 2, 2, 1]), "| đúng:", Solution().longestMonotonicSubarray([3, 2, 2, 1]))

sai: 3 | đúng: 2


**Vì sao sai:** `[3, 2, 2, 1]`: sau `3 → 2` có `dec = 2`; ở cặp `2 → 2` không nhánh nào chạy nên `dec` **giữ `2`**; rồi `2 → 1` làm `dec = 3`, trong khi thực tế `[2, 2, 1]` không giảm ngặt. **Cách sửa:** thêm nhánh `else: inc = dec = 1`.

### Lỗi 3: Khởi tạo bộ đếm bằng `0` thay vì `1`

In [24]:
def sai_khoi_tao_0(nums):
    inc = dec = best = 0                  # SAI: mảng một phần tử vẫn dài 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            inc, dec = inc + 1, 0
        elif nums[i] < nums[i - 1]:
            inc, dec = 0, dec + 1
        else:
            inc = dec = 0
        best = max(best, inc, dec)
    return best


print("sai:", sai_khoi_tao_0([5]), "| đúng:", Solution().longestMonotonicSubarray([5]))
print("sai:", sai_khoi_tao_0([1, 4, 3, 3, 2]), "| đúng:", Solution().longestMonotonicSubarray([1, 4, 3, 3, 2]))

sai: 0 | đúng: 1
sai: 1 | đúng: 2


**Vì sao sai:** một phần tử đứng một mình đã là đoạn dài `1`; khởi tạo `0` làm mọi độ dài thiếu `1` (ví dụ `[1, 4, 3, 3, 2]` cho `1` thay vì `2`, và `[5]` cho `0`). **Cách sửa:** khởi tạo `1`, và reset về `1` (không phải `0`).

### Lỗi 4: Nhầm mảng con với dãy con

In [25]:
def sai_day_con_tang(nums):
    # độ dài dãy con tăng dài nhất (LIS), KHÔNG yêu cầu liên tiếp
    best = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                best[i] = max(best[i], best[j] + 1)
    return max(best)


print("sai:", sai_day_con_tang([1, 3, 2, 4]), "| đúng:", Solution().longestMonotonicSubarray([1, 3, 2, 4]))

sai: 3 | đúng: 2


**Vì sao sai:** `[1, 3, 2, 4]` có dãy con tăng `[1, 3, 4]` (dài `3`) nhưng không phải mảng con (bỏ qua `2`). Đề hỏi mảng con (liên tiếp) nên đáp án là `2`. **Cách sửa:** chỉ so hai phần tử **liền kề**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mảng con **liên tiếp** dài nhất thỏa tính chất cục bộ giữa hai phần tử kề nhau" ⇒ **một lượt với bộ đếm có reset** (`cur = cur + 1 if điều_kiện else 1`).
- Khi có **hai** kiểu (tăng và giảm) ⇒ giữ hai bộ đếm hoặc đảo dấu để tái sử dụng một hàm.
- Luôn kiểm tra **ngặt hay không ngặt** ngay từ đề; thử input toàn phần tử bằng nhau.
- Đoạn gồm `m` bước cùng hướng có `m + 1` phần tử: đừng quên `+ 1`.
- Khởi tạo bằng `1` (một phần tử luôn là một đoạn hợp lệ) và `best = 1`.

## 15. Biến thể của bài

1. **Chỉ xét tăng ngặt** (hoặc chỉ giảm ngặt).
2. **Không ngặt** (non-decreasing hoặc non-increasing): đổi `>` thành `>=`.
3. **Trả về chính mảng con** (hoặc chỉ số đầu/cuối).
4. **Đếm số mảng con đơn điệu ngặt**: mỗi đoạn dài `L` đóng góp `L(L+1)/2` mảng con.
5. **Longest Continuous Increasing Subsequence (674)**, **Monotonic Array (896)**, **Longest Turbulent Subarray (978)**.

In [26]:
def longest_non_strict(nums):
    inc = dec = best = 1
    for i in range(1, len(nums)):
        inc = inc + 1 if nums[i] >= nums[i - 1] else 1
        dec = dec + 1 if nums[i] <= nums[i - 1] else 1
        best = max(best, inc, dec)
    return best


def longest_monotonic_range(nums):
    best, best_range = 1, (0, 0)
    inc = dec = 1
    for i in range(1, len(nums)):
        inc = inc + 1 if nums[i] > nums[i - 1] else 1
        dec = dec + 1 if nums[i] < nums[i - 1] else 1
        for length in (inc, dec):
            if length > best:
                best, best_range = length, (i - length + 1, i)
    return best_range


print(longest_non_strict([3, 3, 3, 3]), longest_non_strict([1, 4, 3, 3, 2]))    # 4 4
print(longest_monotonic_range([1, 2, 3, 2, 1, 0, 0, 5]))                         # (2, 5)
assert longest_non_strict([3, 3, 3, 3]) == 4
assert longest_non_strict([1, 4, 3, 3, 2]) == 4          # 4, 3, 3, 2 không tăng
assert longest_monotonic_range([1, 2, 3, 2, 1, 0, 0, 5]) == (2, 5)

4 4
(2, 5)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra mọi mảng con | định nghĩa | `O(n³)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. Hai lượt (đảo dấu) | một hàm cho cả hai chiều | `O(n)` | `O(n)` | khi muốn tái sử dụng hàm |
| 3. Một lượt, `inc` và `dec` | hai bộ đếm có reset | `O(n)` | `O(1)` | **nên nộp** |
| 4. Trạng thái hướng | `d` và `length` | `O(n)` | `O(1)` | khi cần theo dõi đổi hướng |
| 5. `groupby` trên dấu | các đoạn dấu giống nhau | `O(n)` | `O(n)` | viết ngắn kiểu Pythonic |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(zip([1, 2, 3], [2, 3]))`, `(5 > 3) - (5 < 3)`, `max(2, 5, 3)`, `Solution().longestMonotonicSubarray([1, 2, 2, 3])`, `Solution().longestMonotonicSubarray([9, 7, 5, 6, 8, 10])`.

**Bài 2 (tính tay).** Với `nums = [4, 5, 6, 3, 2, 2, 1, 7, 8]`: lập bảng `i, hướng, inc, dec, best` và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thành `longest_todo(nums)` theo Cách 3.

In [27]:
def longest_todo(nums):
    inc = dec = best = 1
    for i in range(1, len(nums)):
        # TODO 1: nếu nums[i] > nums[i - 1]: inc tăng 1, dec về 1
        # TODO 2: ngược lại nếu nums[i] < nums[i - 1]: dec tăng 1, inc về 1
        # TODO 3: còn lại (bằng nhau): cả hai về 1
        # TODO 4: cập nhật best = max(best, inc, dec)
        pass
    return best

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `longest_groupby2(nums)` theo Cách 5 **không dùng `zip`**: tạo danh sách dấu bằng vòng `for i in range(1, len(nums))`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_monotonic_subarrays(nums)` đếm **số mảng con** (tính theo vị trí) tăng ngặt hoặc giảm ngặt, **mỗi mảng con một phần tử đếm một lần** (mảng con một phần tử đếm `1` lần, không đếm trùng "vừa tăng vừa giảm"). (b) Viết `longest_turbulent(nums)`: mảng con **dài nhất mà dấu của hiệu hai phần tử liền kề luân phiên** (tăng, giảm, tăng, ...), không có cặp bằng nhau.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[(1, 2), (2, 3)]`, `1`, `5`, `2` (`[1, 2]` hoặc `[2, 3]`), `4` (`[5, 6, 8, 10]` tăng ngặt, dài hơn `[9, 7, 5]` dài `3`). Bài 3: bốn chỗ TODO, nhánh `else` reset cả hai.
- **Mức 2 (vừa):** Bài 2: hướng các bước: tăng, tăng, giảm, giảm, bằng, giảm, tăng, tăng; đoạn tăng dài nhất `[4, 5, 6]` (3), đoạn giảm dài nhất `[6, 3, 2]` (3); `[2, 1]`(2); `[1, 7, 8]` (3). Đáp án `3`. Bài 4: `signs.append(sign(nums[i] - nums[i - 1]))`.
- **Mức 3 (sâu):** Bài 5(a): đoạn đơn điệu ngặt dài `L` có `L(L+1)/2` mảng con; nhưng một mảng con một phần tử thuộc cả hai nhóm tăng/giảm nên cần đếm cẩn thận: đếm số mảng con độ dài `1` (n) cộng số mảng con độ dài `>= 2` tăng hoặc giảm (không thể vừa tăng vừa giảm). 5(b): theo dõi dấu bước trước `prev` và luân phiên: nếu `s * prev < 0` thì kéo dài, nếu `s != 0` mà không luân phiên thì `length = 2`, nếu `s == 0` thì `length = 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert list(zip([1, 2, 3], [2, 3])) == [(1, 2), (2, 3)]
assert (5 > 3) - (5 < 3) == 1
assert max(2, 5, 3) == 5
assert Solution().longestMonotonicSubarray([1, 2, 2, 3]) == 2
assert Solution().longestMonotonicSubarray([9, 7, 5, 6, 8, 10]) == 4

# Bài 2
assert Solution().longestMonotonicSubarray([4, 5, 6, 3, 2, 2, 1, 7, 8]) == 3


# Bài 3
def longest_todo_ans(nums):
    inc = dec = best = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            inc, dec = inc + 1, 1
        elif nums[i] < nums[i - 1]:
            inc, dec = 1, dec + 1
        else:
            inc = dec = 1
        best = max(best, inc, dec)
    return best


for nums, expected in examples:
    assert longest_todo_ans(nums) == expected


# Bài 4
def longest_groupby2(nums):
    signs = []
    for i in range(1, len(nums)):
        signs.append(sign(nums[i] - nums[i - 1]))
    best = 1
    for s, grp in groupby(signs):
        if s != 0:
            best = max(best, len(list(grp)) + 1)
    return best


for nums, expected in examples:
    assert longest_groupby2(nums) == expected


# Bài 5(a)
def count_monotonic_subarrays(nums):
    n = len(nums)
    total = n                                   # mỗi mảng con một phần tử
    for i in range(n):
        for j in range(i + 1, n):
            sub = nums[i:j + 1]
            if is_strictly_increasing(sub) or is_strictly_decreasing(sub):
                total += 1
    return total


assert count_monotonic_subarrays([1, 2, 3]) == 3 + 3        # 3 đơn lẻ + [1,2],[2,3],[1,2,3]
assert count_monotonic_subarrays([3, 3]) == 2


# Bài 5(b)
def longest_turbulent(nums):
    best = length = 1
    prev = 0
    for i in range(1, len(nums)):
        s = sign(nums[i] - nums[i - 1])
        if s == 0:
            length, prev = 1, 0
        elif prev != 0 and s * prev < 0:
            length += 1
            prev = s
        else:
            length, prev = 2, s
        best = max(best, length)
    return best


assert longest_turbulent([9, 4, 2, 10, 7, 8, 8, 1, 9]) == 5      # 4, 2, 10, 7, 8
assert longest_turbulent([4, 8, 12, 16]) == 2
assert longest_turbulent([100]) == 1
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mảng con liên tiếp dài nhất theo tính chất giữa hai phần tử kề nhau)

```python
from typing import List

class Solution:
    def longestMonotonicSubarray(self, nums: List[int]) -> int:
        inc = dec = best = 1
        for i in range(1, len(nums)):
            if nums[i] > nums[i - 1]:
                inc, dec = inc + 1, 1
            elif nums[i] < nums[i - 1]:
                inc, dec = 1, dec + 1
            else:
                inc = dec = 1
            best = max(best, inc, dec)
        return best
```

**Checklist**

- [ ] Dùng so sánh **ngặt** (`>`, `<`), và hai phần tử **bằng nhau** reset cả hai bộ đếm về `1`?
- [ ] Khởi tạo `inc = dec = best = 1` (không phải `0`)?
- [ ] Cập nhật `best` ở **mỗi bước** (kể cả đoạn kết thúc ở phần tử cuối)?
- [ ] Hiểu đề hỏi **mảng con (liên tiếp)**, không phải dãy con?
- [ ] Đã thử `[5]`, `[3, 3, 3, 3]`, mảng tăng toàn bộ và giảm toàn bộ?
- [ ] Skeleton đúng `class Solution` / `def longestMonotonicSubarray(self, nums: List[int]) -> int`?